**Table of contents**<a id='toc0_'></a>    
- [Antibunching: 4 fluorophores, different distances, OET high SSA](#toc1_)    
  - [Preparation](#toc1_1_)    
  - [Generation](#toc1_2_)    

<!-- vscode-jupyter-toc-config
	numbering=false
	anchor=true
	flat=false
	minLevel=1
	maxLevel=6
	/vscode-jupyter-toc-config -->
<!-- THIS CELL WILL BE REPLACED ON TOC UPDATE. DO NOT WRITE YOUR TEXT IN THIS CELL -->

# <a id='toc1_'></a>[Antibunching: 4 fluorophores, different distances, OET high SSA](#toc0_)

In [ ]:
import numpy as np

import fluopy.emissions as em
import fluopy.fluorophores as fl
import fluopy.routines as rt
import fluopy.transitions as tr

%load_ext autoreload
%autoreload 2

saving_at = r"D:\python_output\Chapter_I\1_12_multi_f_antibunching\OEThighSSA"

## <a id='toc1_1_'></a>[Preparation](#toc0_)

In [ ]:
def generate_data(distance):
    PARAMS_ADJ = {
        k: v for k, v in rt.PARAMS_DSTORM.items() if k != "energy_transfer_parameters"
    }

    fluorophores = fl.construct_fluorophores(
        name="cy5_dna", count=4, distance=distance, shape="square"
    )
    fluorophore_system = fl.FluorophoreSystem(fluorophores=fluorophores)
    transitions = fluorophore_system.load_transitions(
        bleaching=True,
        summarize=True,
        energy_transfer=True,
        **PARAMS_ADJ,
        energy_transfer_parameters={
            "overwrite": {"off": [1, 1e-4], "s1": [1, 0]},
            "exclude": ["s0"],
        },
    )
    transition_set = tr.TransitionSet(transitions, fluorophore_system)
    transition_set.finalize()

    rng = np.random.default_rng(1)
    emis = em.Emissions(seed=rng, **rt.PARAMS_EMIS)
    _, _, tau, _ = emis.tcspc(
        transition_set, number_pulses=4e9, **rt.PARAMS_PULSE, store_time_points=True
    )
    emis.event_time_series.to_frame().to_parquet(
        saving_at + rf"\event_time_series_{distance}nm.parquet"
    )
    np.save(saving_at + rf"\tau_{distance}nm.npy", tau)
    np.save(saving_at + rf"\event_time_points_{distance}nm.npy", emis.event_time_points)

## <a id='toc1_2_'></a>[Generation](#toc0_)

In [ ]:
generate_data(3)
generate_data(6)
generate_data(9)
generate_data(18)

WARNING for line:             warnings.warn(
 The irradiance used initially for excitation rates in
 transition_set is now assumed to be the mean irradiance of
 pulse and no pulse duration. 
WARNING for line:     warnings.warn(
 the last frame (of index 0.025) has 1.00e+00 times the pulses of other frames. 
